# Janome で形態素解析する (Pure Python)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/04_japanese_tokenization/janome.ipynb)

NLP クックブック 04-14 ／ 日本語の形態素解析と文分割

C 拡張なしで動く解析器を使いたい。フィルタを組み合わせた前処理パイプラインも作りたい。

In [1]:
!pip install -q janome

In [2]:
from janome.tokenizer import Tokenizer

t = Tokenizer()
for token in t.tokenize("すもももももももものうち"):
    print(token.surface, token.part_of_speech, token.base_form, token.reading, sep="\t")
print(list(t.tokenize("分かち書きもできます", wakati=True)))

すもも	名詞,一般,*,*	すもも	スモモ
も	助詞,係助詞,*,*	も	モ
もも	名詞,一般,*,*	もも	モモ
も	助詞,係助詞,*,*	も	モ
もも	名詞,一般,*,*	もも	モモ
の	助詞,連体化,*,*	の	ノ
うち	名詞,非自立,副詞可能,*	うち	ウチ
['分かち書き', 'も', 'でき', 'ます']


In [3]:
# Analyzer: 文字フィルタ → トークナイザ → トークンフィルタ のパイプライン
from janome.analyzer import Analyzer
from janome.charfilter import UnicodeNormalizeCharFilter, RegexReplaceCharFilter
from janome.tokenfilter import POSKeepFilter, LowerCaseFilter, ExtractAttributeFilter

analyzer = Analyzer(
    char_filters=[UnicodeNormalizeCharFilter(), RegexReplaceCharFilter(r"https?://\S+", "")],
    tokenizer=Tokenizer(),
    token_filters=[POSKeepFilter(["名詞", "動詞", "形容詞"]), LowerCaseFilter(), ExtractAttributeFilter("base_form")],
)
print(list(analyzer.analyze("Ｊａｎｏｍｅは遅いけれど手軽に使えて嬉しい https://example.com")))

['janome', '遅い', '手軽', '使える', '嬉しい']


- 辞書は IPA 辞書。速度は MeCab の 10 分の 1 程度なので、大規模データには向かない。